# Persona 2 - Recuperacion Clasica (TF-IDF y BM25)
### Proyecto Bimestre 1 - Recuperacion de Informacion

## 1. Importaciones y configuracion

In [ ]:
import pandas as pd
import numpy as np
import nltk

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import defaultdict, Counter

In [ ]:
import string
import math

In [ ]:
# Descarga de recursos necesarios de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

## 2. Carga del corpus
Se utiliza el dataset Reuters-21578 (ModApte split).  
Se eliminan las filas que no tienen texto.

In [ ]:
# Cargar el dataset
df = pd.read_csv("../../ModApte_test.csv")

# Eliminar documentos sin texto
df = df.dropna(subset=['text'])

print(f"Documentos cargados: {len(df)}")
print(f"Columnas disponibles: {df.columns.tolist()}")

In [ ]:
df.head(3)

In [ ]:
df.tail(3)

## 3. Preprocesamiento
Se aplica a cada documento:
- Conversion a minusculas
- Tokenizacion
- Eliminacion de signos de puntuacion
- Eliminacion de stopwords en ingles
- Eliminacion de tokens cortos (menor a 3 caracteres)

In [ ]:
# Cargar stopwords en ingles
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Convertir a minusculas
    text = text.lower()

    # Tokenizar el texto
    tokens = word_tokenize(text)

    # Limpiar tokens: eliminar puntuacion, stopwords y tokens cortos
    tokens = [
        t.replace('.', '').replace('-', ' ').strip()
        for t in tokens
        if t not in string.punctuation
        and t not in stop_words
        and len(t) > 2
    ]

    # Eliminar tokens vacios que quedaron tras el reemplazo
    tokens = [t for t in tokens if t]

    return tokens

# Verificar con el primer documento
print('Ejemplo de tokens del primer documento:')
print(preprocess(df['text'].iloc[0])[:20])

## 4. Indice invertido y estadisticas
Se construyen estructuras para recuperacion clasica:
- indice invertido (termino -> {doc_id: frecuencia})
- frecuencia de documento (df)
- longitud de documento y promedio

In [ ]:
# Tokenizar todos los documentos
df['tokens'] = df['text'].apply(preprocess)

In [ ]:
# Construccion del indice invertido
# termino -> {doc_id: frecuencia}
inverted_index = defaultdict(dict)
doc_len = {}

for i, tokens in enumerate(df['tokens']):
    term_counts = Counter(tokens)
    doc_len[i] = sum(term_counts.values())
    for term, freq in term_counts.items():
        inverted_index[term][i] = freq

doc_freq = {term: len(postings) for term, postings in inverted_index.items()}
N = len(df)
avg_doc_len = sum(doc_len.values()) / N

print(f'Documentos indexados: {N}')
print(f'Terminos en vocabulario: {len(inverted_index)}')
print(f'Longitud promedio de documento: {avg_doc_len:.2f} tokens')

In [ ]:
# Verificar estructura del indice invertido
# Para cada termino: documentos en los que aparece + frecuencia
sample_terms = list(inverted_index.keys())[:10]

rows = []
for term in sample_terms:
    postings = inverted_index[term]
    df_t = doc_freq[term]
    for doc_id, freq in list(postings.items())[:3]:
        rows.append({'termino': term, 'doc_id': doc_id, 'frecuencia': freq, 'df (docs con termino)': df_t})

pd.DataFrame(rows)

## 5. Modelos clasicos de recuperacion
Se implementan dos modelos:
- TF-IDF + similitud coseno
- BM25

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Vectorizar usando tokens preprocesados (mismos que el indice invertido)
corpus = df['tokens'].apply(lambda tokens: ' '.join(tokens)).tolist()

vectorizer = TfidfVectorizer()
matriz_tfidf = vectorizer.fit_transform(corpus)

print(f"Forma de la matriz TF-IDF: {matriz_tfidf.shape}")
print(f"  {matriz_tfidf.shape[0]} documentos x {matriz_tfidf.shape[1]} terminos")

table = pd.DataFrame(
    matriz_tfidf.toarray(),
    index=[f"doc{i+1}" for i in range(matriz_tfidf.shape[0])],
    columns=vectorizer.get_feature_names_out()
)

print(table.head(10))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search_tfidf_cosine(query, top_k=10):
    # Preprocesar la consulta con el mismo pipeline del corpus
    query_tokens = preprocess(query)
    query_text = ' '.join(query_tokens)

    # Vectorizar la consulta usando el mismo vectorizer entrenado en el corpus
    query_vector = vectorizer.transform([query_text])

    # Calcular similitud coseno entre la consulta y todos los documentos
    similitudes = cosine_similarity(query_vector, matriz_tfidf).flatten()

    # Crear ranking ordenado por similitud descendente (solo docs con score > 0)
    ranking = sorted(enumerate(similitudes), key=lambda x: x[1], reverse=True)
    return [(indice_doc, score) for indice_doc, score in ranking[:top_k] if score > 0]

def search_bm25(query, top_k=10, k1=1.5, b=0.75):
    query_tokens = preprocess(query)
    frecuencia_query = Counter(query_tokens)
    scores = defaultdict(float)

    for termino, freq_en_query in frecuencia_query.items():
        postings = inverted_index.get(termino)
        if not postings:
            continue
        frecuencia_documento = doc_freq[termino]
        idf = math.log(1.0 + ((N - frecuencia_documento + 0.5) / (frecuencia_documento + 0.5)))
        for indice_doc, freq_en_doc in postings.items():
            longitud_doc = doc_len[indice_doc]
            denominador = freq_en_doc + k1 * (1.0 - b + b * (longitud_doc / avg_doc_len))
            scores[indice_doc] += idf * ((freq_en_doc * (k1 + 1.0)) / denominador)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [ ]:
def search_jaccard(query, top_k=10):
    # Preprocesar la consulta
    query_tokens = set(preprocess(query))

    if not query_tokens:
        return []

    scores = {}

    for indice_doc, tokens in enumerate(df['tokens']):
        conjunto_documento = set(tokens)

        # Similitud Jaccard = |interseccion| / |union|
        interseccion = query_tokens & conjunto_documento
        union = query_tokens | conjunto_documento

        if len(union) > 0:
            similitud = len(interseccion) / len(union)
            if similitud > 0:
                scores[indice_doc] = similitud

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

## 6. Consulta de texto libre y ranking
Se ejecuta una consulta y se muestran los resultados ordenados por relevancia.

In [ ]:
def show_results(ranked_docs, model_name='Modelo', top_n=5):
    print(f'\n[{model_name}] Top {min(top_n, len(ranked_docs))} resultados')

    if not ranked_docs:
        print('No se encontraron resultados.')
        return

    for rank, (doc_id, score) in enumerate(ranked_docs[:top_n], start=1):
        row = df.iloc[doc_id]
        title = str(row['title']) if pd.notna(row['title']) else ''
        text = str(row['text']).replace('\n', ' ')
        snippet = text[:140] + ('...' if len(text) > 140 else '')

        print(f'{rank}. idx={doc_id} | score={score:.5f}')
        if title:
            print(f'   title: {title}')
        print(f'   text : {snippet}')

query = 'trade japan market'

tfidf_results  = search_tfidf_cosine(query, top_k=10)
bm25_results   = search_bm25(query, top_k=10)
jaccard_results = search_jaccard(query, top_k=10)

print(f'Consulta: {query}')

In [ ]:
show_results(tfidf_results,   model_name='TF-IDF + Coseno', top_n=5)

In [ ]:
show_results(bm25_results,    model_name='BM25', top_n=5)

In [ ]:
show_results(jaccard_results, model_name='Jaccard', top_n=5)

## 7. Evaluacion de resultados
Se usan los topics del corpus como qrels:
- Cada topic unico es una consulta de prueba
- Documentos relevantes = todos los docs que contienen ese topic
- Metricas: Precision@k, Recall@k y MAP

In [ ]:
import re

# Construir qrels: topic -> conjunto de indices de documentos relevantes
# Los topics tienen formato numpy: ['crude' 'nat-gas'] (sin coma entre items)
# Solucion: regex para extraer cada string entre comillas individualmente
qrels = defaultdict(set)

for i, row in df.iterrows():
    valor_topics = row['topics']
    if pd.isna(valor_topics):
        continue
    # Extraer todos los strings entre comillas simples
    lista_topics = re.findall(r"'([^']+)'", str(valor_topics))
    for topic in lista_topics:
        topic = topic.strip()
        if topic:
            qrels[topic].add(df.index.get_loc(i))

# Solo topics con al menos 2 documentos relevantes
qrels = {topic: docs for topic, docs in qrels.items() if len(docs) >= 2}

print(f"Topics usados como consultas: {len(qrels)}")
print(f"Ejemplo de topics: {list(qrels.keys())[:10]}")
print(f"Docs relevantes para 'trade': {len(qrels.get('trade', set()))}")
print(f"Docs relevantes para 'crude': {len(qrels.get('crude', set()))}")
print(f"Docs relevantes para 'nat-gas': {len(qrels.get('nat-gas', set()))}")

In [ ]:
def precision_at_k(ranked_docs, relevant_docs, k):
    """Fraccion de documentos relevantes en los primeros k resultados."""
    top_k = [doc_id for doc_id, _ in ranked_docs[:k]]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_docs)
    return hits / k if k > 0 else 0.0


def recall_at_k(ranked_docs, relevant_docs, k):
    """Fraccion de relevantes recuperados en los primeros k resultados."""
    top_k = [doc_id for doc_id, _ in ranked_docs[:k]]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_docs)
    return hits / len(relevant_docs) if len(relevant_docs) > 0 else 0.0


def average_precision(ranked_docs, relevant_docs):
    """Average Precision para una sola consulta."""
    if not relevant_docs:
        return 0.0
    hits = 0
    precision_sum = 0.0
    for rank, (doc_id, _) in enumerate(ranked_docs, start=1):
        if doc_id in relevant_docs:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / len(relevant_docs)


def evaluar_modelo(search_fn, qrels, k=10):
    """
    Evalua un modelo sobre todas las consultas en qrels.
    Retorna Precision@k, Recall@k y MAP promedio.
    """
    precision_scores = []
    recall_scores = []
    ap_scores = []

    for topic, relevant_docs in qrels.items():
        ranked_docs = search_fn(topic, top_k=k)
        precision_scores.append(precision_at_k(ranked_docs, relevant_docs, k))
        recall_scores.append(recall_at_k(ranked_docs, relevant_docs, k))
        ap_scores.append(average_precision(ranked_docs, relevant_docs))

    return {
        f'Precision@{k}': round(sum(precision_scores) / len(precision_scores), 4),
        f'Recall@{k}':    round(sum(recall_scores)    / len(recall_scores),    4),
        'MAP':            round(sum(ap_scores)         / len(ap_scores),        4),
    }

print("Funciones de evaluacion listas.")

In [ ]:
# Evaluar los tres modelos con k=10
K = 10

print(f"Evaluando modelos con {len(qrels)} consultas (topics) y k={K}...")
print("Esto puede tardar unos segundos (Jaccard itera sobre todo el corpus).\n")

resultados_tfidf   = evaluar_modelo(search_tfidf_cosine, qrels, k=K)
resultados_bm25    = evaluar_modelo(search_bm25,         qrels, k=K)
resultados_jaccard = evaluar_modelo(search_jaccard,      qrels, k=K)

# Tabla comparativa
tabla_evaluacion = pd.DataFrame(
    [resultados_tfidf, resultados_bm25, resultados_jaccard],
    index=['TF-IDF + Coseno', 'BM25', 'Jaccard']
)

print("=== Comparacion de modelos ===")
print(tabla_evaluacion.to_string())
tabla_evaluacion

## 8. Recuperación semántica con Embeddings
Se utiliza un modelo preentrenado (`all-MiniLM-L6-v2`) de `sentence-transformers` para generar embeddings densos.  
Los vectores se almacenan en **ChromaDB** para búsqueda eficiente por similitud coseno.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Modelo ligero y eficaz para recuperacion semantica
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(EMBEDDING_MODEL)

print(f"Modelo cargado: {EMBEDDING_MODEL}")

In [ ]:
# Generar embeddings para todos los documentos del corpus
# Se usa el texto original (no tokenizado) para aprovechar el contexto del modelo
print("Generando embeddings para el corpus (puede tardar ~1-2 min)...")

corpus_texts = df['text'].tolist()
doc_embeddings = embedder.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generados: {doc_embeddings.shape}")
print(f"  {doc_embeddings.shape[0]} documentos x {doc_embeddings.shape[1]} dimensiones")

In [ ]:
# Crear coleccion ChromaDB en memoria
chroma_client = chromadb.Client()

# Eliminar coleccion si existe (re-ejecucion del notebook)
try:
    chroma_client.delete_collection("reuters_corpus")
except:
    pass

collection = chroma_client.create_collection(
    name="reuters_corpus",
    metadata={"hnsw:space": "cosine"}
)

# Insertar documentos en lotes (ChromaDB tiene limite por batch)
BATCH_SIZE = 500
ids     = [str(i) for i in range(len(df))]
docs    = corpus_texts
embeds  = doc_embeddings.tolist()

for start in range(0, len(ids), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(ids))
    collection.add(
        ids=ids[start:end],
        documents=docs[start:end],
        embeddings=embeds[start:end]
    )

print(f"ChromaDB: {collection.count()} documentos indexados")

In [ ]:
def search_embeddings(query, top_k=10):
    """Busqueda semantica usando ChromaDB + sentence-transformers."""
    query_embedding = embedder.encode([query], normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    # Distancia coseno ChromaDB: 0 = identico, 2 = opuesto
    # Convertir a similitud: score = 1 - distancia
    doc_ids    = [int(i) for i in results['ids'][0]]
    distances  = results['distances'][0]
    scores     = [1.0 - d for d in distances]
    return list(zip(doc_ids, scores))

# Prueba rapida
query_test = 'trade japan market'
emb_results = search_embeddings(query_test, top_k=10)
show_results(emb_results, model_name='Embeddings (ChromaDB)', top_n=5)

In [ ]:
# Evaluar modelo de embeddings con las mismas metricas y qrels
print("Evaluando modelo de embeddings...")
resultados_emb = evaluar_modelo(search_embeddings, qrels, k=K)
print(f"Embeddings: {resultados_emb}")

## 9. Comparación de todos los modelos
Se comparan los cuatro modelos de recuperación:
- **Jaccard** — similitud binaria por conjuntos
- **TF-IDF + Coseno** — modelo vectorial clásico
- **BM25** — modelo probabilístico con normalización de longitud
- **Embeddings semánticos** — representación densa con modelo preentrenado + ChromaDB

In [ ]:
# Tabla comparativa de los 4 modelos
tabla_final = pd.DataFrame(
    [resultados_jaccard, resultados_tfidf, resultados_bm25, resultados_emb],
    index=['Jaccard', 'TF-IDF + Coseno', 'BM25', 'Embeddings (ChromaDB)']
)

print(f"=== Comparacion final de modelos (k={K}, {len(qrels)} consultas) ===")
print(tabla_final.to_string())
tabla_final

In [ ]:
import matplotlib.pyplot as plt

# Grafico de barras comparativo
metricas = [f'Precision@{K}', f'Recall@{K}', 'MAP']
modelos  = tabla_final.index.tolist()
x = np.arange(len(metricas))
ancho = 0.18

fig, ax = plt.subplots(figsize=(10, 5))
for i, modelo in enumerate(modelos):
    vals = [tabla_final.loc[modelo, m] for m in metricas]
    ax.bar(x + i * ancho, vals, ancho, label=modelo)

ax.set_xticks(x + ancho * 1.5)
ax.set_xticklabels(metricas, fontsize=12)
ax.set_ylabel('Score')
ax.set_title(f'Comparacion de modelos de recuperacion (k={K})')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Análisis de resultados

**Observaciones esperadas:**
- **TF-IDF + Coseno** y **BM25** destacan en consultas de términos exactos del corpus (palabras clave del dominio financiero/noticias Reuters).
- **Embeddings semánticos** capturan paráfrasis y sinónimos que los modelos léxicos ignoran (ej. "oil price drop" → recupera docs sobre "petroleum cost decrease").
- **Jaccard** es el modelo más débil al no ponderar frecuencias ni contexto semántico.
- Los embeddings pueden fallar cuando los *topics* del qrel son términos muy específicos del dominio (ej. `nat-gas`, `veg-oil`) que el modelo preentrenado no asocia directamente.

## 10. Interfaz CLI interactiva
Permite realizar consultas de texto libre seleccionando el modelo de recuperación.  
Escribe `salir` para terminar.

In [ ]:
print("=== Sistema de Recuperacion de Informacion ===")
print("Modelos disponibles: tfidf | bm25 | jaccard | embeddings")
print("Escribe 'salir' para terminar\n")

while True:
    query = input("Consulta: ").strip()
    if query.lower() == 'salir':
        print("Saliendo...")
        break
    if not query:
        continue

    modelo = input("Modelo [tfidf/bm25/jaccard/embeddings]: ").strip().lower()

    if modelo == 'tfidf':
        resultados = search_tfidf_cosine(query, top_k=10)
        show_results(resultados, model_name='TF-IDF + Coseno')
    elif modelo == 'bm25':
        resultados = search_bm25(query, top_k=10)
        show_results(resultados, model_name='BM25')
    elif modelo == 'jaccard':
        resultados = search_jaccard(query, top_k=10)
        show_results(resultados, model_name='Jaccard')
    elif modelo == 'embeddings':
        resultados = search_embeddings(query, top_k=10)
        show_results(resultados, model_name='Embeddings (ChromaDB)')
    else:
        print(f"Modelo '{modelo}' no reconocido. Usa: tfidf | bm25 | jaccard | embeddings")
    print()